In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import pandas as pd
from scipy.interpolate import PchipInterpolator
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import os

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.unicode_minus"] = False

# ================== 输出目录 ==================
output_dir = r"E:\关中干旱小论文\研究区域图-图一"
os.makedirs(output_dir, exist_ok=True)

# ================== 基础设置 ==================
years_clim  = range(1991, 2021)
year_target = 2025

start_date = pd.to_datetime("2025-03-01")
end_date   = pd.to_datetime("2025-05-31")

lon_min, lon_max = 100, 180
lat_min, lat_max = 10, 60

lon_slice = slice(lon_min, lon_max)
lat_slice_hgt = slice(lat_max, lat_min)

level_use = 500

# ================== 时间分段 ==================
date_ranges = []
cur = start_date
while cur <= end_date:
    pe = min(cur + timedelta(days=14), end_date)
    date_ranges.append((cur, pe))
    cur += timedelta(days=15)
date_ranges = date_ranges[:6]

lon_ticks_sparse = [100, 120, 140, 160, 180]
lat_ticks_sparse = [10, 30, 50, 60]

# ================== HGT 背景场设置 ==================
hgt_levels = np.arange(5000, 5920 + 40, 40)
HGT_VMIN = float(hgt_levels.min())
HGT_VMAX = float(hgt_levels.max())

hgt_cmap = plt.get_cmap("RdYlBu_r")
hgt_norm = mcolors.Normalize(vmin=HGT_VMIN, vmax=HGT_VMAX)
hgt_alpha = 0.60

# ================== 东亚槽槽线参数 ==================
EA_LON_MIN, EA_LON_MAX = 110, 170
EA_LAT_MIN, EA_LAT_MAX = 30, 55

CLIM_COLOR = "#D00000"
YR_COLOR   = "#00A000"
CLIM_LS = "--"
YR_LS   = "-"
CLIM_LW = 2.20
YR_LW   = 2.50

ALLOW_FALLBACK_GLOBALMIN = False
JUMP_MAX_DEG = 4.5
W_JUMP = 8.0
W_Z = 1.0
W_CURV = 4.0
POST_MAX_STEP = 4.5
MEDIAN_WIN = 5

# ================== 核心函数 ==================
def open_hgt_year(year: int) -> xr.DataArray:
    fp = f"E:/hgt/hgt.{year}.nc"
    da = xr.open_dataset(fp)["hgt"]
    da = da.sel(level=level_use, lon=lon_slice, lat=lat_slice_hgt)
    da = da.sel(time=da.time.dt.month.isin([3, 4, 5]))
    if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0):
        da = da.where(~((da.time.dt.month == 2) & (da.time.dt.day == 29)), drop=True)
    return da

def mean_period_by_slice(ds, start, end):
    sub = ds.sel(time=slice(start, end))
    return sub.mean("time") if sub.time.size > 0 else None

def mean_period_by_monthday(ds_all, start, end):
    t = ds_all.time
    cond = (
        ((t.dt.month == start.month) & (t.dt.day >= start.day)) |
        ((t.dt.month >  start.month) & (t.dt.month < end.month)) |
        ((t.dt.month == end.month)   & (t.dt.day <= end.day))
    )
    sub = ds_all.sel(time=cond)
    return sub.mean("time") if sub.time.size > 0 else None

def _row_local_minima_candidates(row, lons):
    v = np.asarray(row, dtype=float).copy()
    valid = np.isfinite(v)
    if np.count_nonzero(valid) < 3: return []
    v[~valid] = np.inf
    cand = []
    for i in range(1, len(v) - 1):
        if np.isfinite(v[i]) and (v[i] < v[i - 1]) and (v[i] < v[i + 1]):
            cand.append((i, float(lons[i]), float(v[i])))
    return cand

def _median_filter_1d(x, win=5):
    x = np.array(x, dtype=float)
    if win < 3 or win % 2 == 0: return x
    half = win // 2
    y = x.copy()
    for i in range(len(x)):
        a = max(0, i - half)
        b = min(len(x), i + half + 1)
        seg = x[a:b]
        seg = seg[np.isfinite(seg)]
        if seg.size > 0: y[i] = np.median(seg)
    return y

def _remove_jumps_and_interpolate(lon, lat, max_step=4.5):
    lon = np.array(lon, dtype=float)
    lat = np.array(lat, dtype=float)
    if lon.size < 3: return lon, lat
    bad = np.zeros_like(lon, dtype=bool)
    for i in range(1, len(lon)):
        if np.isfinite(lon[i]) and np.isfinite(lon[i-1]):
            if abs(lon[i] - lon[i-1]) > max_step:
                bad[i] = True
                bad[i-1] = True
    lon2 = lon.copy()
    lon2[bad] = np.nan
    if np.isnan(lon2[0]):
        j = np.where(np.isfinite(lon2))[0]
        if j.size > 0: lon2[0] = lon2[j[0]]
    if np.isnan(lon2[-1]):
        j = np.where(np.isfinite(lon2))[0]
        if j.size > 0: lon2[-1] = lon2[j[-1]]
    ok = np.isfinite(lon2)
    if np.count_nonzero(ok) >= 2:
        lon2[~ok] = np.interp(lat[~ok], lat[ok], lon2[ok])
    return lon2, lat

def extract_ea_trough_line_and_index_optimized(hgt2d, lon_min_box=110, lon_max_box=170, lat_min_box=30, lat_max_box=55):
    da = hgt2d.sortby("lat")
    sub = da.sel(lon=slice(lon_min_box, lon_max_box), lat=slice(lat_min_box, lat_max_box))
    if sub.size == 0 or sub.lon.size < 3 or sub.lat.size < 2: return None, None, np.nan
    lons = sub.lon.values
    lats = sub.lat.values
    cands = []
    lat_used = []
    for la in lats:
        row = sub.sel(lat=la).values
        cand = _row_local_minima_candidates(row, lons)
        if len(cand) == 0 and ALLOW_FALLBACK_GLOBALMIN:
            if np.any(np.isfinite(row)):
                i0 = int(np.nanargmin(row))
                cand = [(i0, float(lons[i0]), float(row[i0]))]
        if len(cand) == 0: continue
        cands.append(cand)
        lat_used.append(float(la))
    if len(cands) < 2: return None, None, np.nan
    lat_used = np.array(lat_used, dtype=float)
    allz = np.array([z for cand in cands for (_, _, z) in cand if np.isfinite(z)], dtype=float)
    zmin, zmax = float(allz.min()), float(allz.max())
    zrange = max(zmax - zmin, 1e-6)
    dp = []
    parent = []
    prev_lons = None
    prevprev_lons = None
    prev_parent = None
    for t, cand in enumerate(cands):
        cur_lons = np.array([x[1] for x in cand], dtype=float)
        cur_z = np.array([x[2] for x in cand], dtype=float)
        m = len(cand)
        z_cost = W_Z * ((cur_z - zmin) / zrange)
        dp_t = np.full(m, np.inf, dtype=float)
        par_t = np.full(m, -1, dtype=int)
        if t == 0:
            dp_t = z_cost.copy()
        else:
            for j in range(m):
                best = np.inf
                bestk = -1
                for k in range(len(prev_lons)):
                    jump = abs(cur_lons[j] - prev_lons[k])
                    if jump > JUMP_MAX_DEG: continue
                    jump_cost = W_JUMP * (jump / JUMP_MAX_DEG)
                    curv_cost = 0.0
                    if t >= 2 and prevprev_lons is not None and prev_parent is not None:
                        kk = prev_parent[k]
                        if kk < 0: kk = int(np.argmin(dp[t-2]))
                        lonkk = prevprev_lons[kk]
                        curv = abs(cur_lons[j] - 2.0 * prev_lons[k] + lonkk)
                        curv_cost = W_CURV * (curv / max(JUMP_MAX_DEG, 1e-6))
                    val = dp[t-1][k] + z_cost[j] + jump_cost + curv_cost
                    if val < best:
                        best = val
                        bestk = k
                if np.isinf(best):
                    k_near = int(np.argmin(np.abs(prev_lons - cur_lons[j])))
                    jump = abs(cur_lons[j] - prev_lons[k_near])
                    best = dp[t-1][k_near] + z_cost[j] + W_JUMP * 2.5 * (jump / max(JUMP_MAX_DEG, 1e-6))
                    bestk = k_near
                dp_t[j] = best
                par_t[j] = bestk
        dp.append(dp_t)
        parent.append(par_t)
        prevprev_lons = prev_lons
        prev_lons = cur_lons
        prev_parent = par_t
    t_last = len(cands) - 1
    j_last = int(np.argmin(dp[t_last]))
    lon_path = []
    lat_path = []
    j = j_last
    for t in range(t_last, -1, -1):
        lon_path.append(float(cands[t][j][1]))
        lat_path.append(float(lat_used[t]))
        j = parent[t][j]
        if j < 0 and t > 0: j = 0
    lon_path = np.array(lon_path[::-1], dtype=float)
    lat_path = np.array(lat_path[::-1], dtype=float)
    lon_f = _median_filter_1d(lon_path, win=MEDIAN_WIN)
    lon_f, lat_f = _remove_jumps_and_interpolate(lon_f, lat_path, max_step=POST_MAX_STEP)
    mean_lon_index = float(np.nanmean(lon_f))
    try:
        order = np.argsort(lat_f)
        lat0 = lat_f[order]
        lon0 = lon_f[order]
        lat_u, idx = np.unique(lat0, return_index=True)
        lon_u = lon0[idx]
        if len(lat_u) >= 4:
            pchip = PchipInterpolator(lat_u, lon_u, extrapolate=False)
            lat_s = np.linspace(lat_u.min(), lat_u.max(), 180)
            lon_s = pchip(lat_s)
            lon_s = np.clip(lon_s, lon_min_box, lon_max_box)
            return lon_s, lat_s, mean_lon_index
    except Exception:
        pass
    return lon_f, lat_f, mean_lon_index

def draw_domain_border(ax, lw=0.9, alpha=0.95):
    xs = [lon_min, lon_max, lon_max, lon_min, lon_min]
    ys = [lat_min, lat_min, lat_max, lat_max, lat_min]
    ax.plot(xs, ys, transform=ccrs.PlateCarree(), color='k', lw=lw, alpha=alpha, zorder=30)

# ================== 单个子图绘图 ==================
def plot_panel(ax, hm_yr, hm_clim, title_text, is_first_panel=False):
    proj = ccrs.PlateCarree()
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=proj)

    if hm_yr is not None:
        da = hm_yr.sortby("lat")
        lons = da.lon.values
        lats = da.lat.values
        A2d  = da.values
        
        # 填色
        ax.contourf(
            lons, lats, A2d,
            levels=hgt_levels, cmap=hgt_cmap, norm=hgt_norm,
            alpha=hgt_alpha, extend='both', transform=proj
        )
        # 等高线
        cs = ax.contour(
            lons, lats, A2d,
            levels=hgt_levels, colors='k', linewidths=0.5, alpha=0.7, transform=proj
        )
        # 优化：隔几个标一个数字，并放大字体
        ax.clabel(cs, cs.levels[::2], inline=True, fontsize=9, fmt='%1.0f')

    ax.add_feature(cfeature.COASTLINE.with_scale("110m"), linewidth=0.35, edgecolor="k", alpha=0.9)

    # 优化：只给第一张图画经纬度刻度
    if is_first_panel:
        ax.set_xticks(lon_ticks_sparse, crs=proj)
        ax.set_yticks(lat_ticks_sparse, crs=proj)
        ax.xaxis.set_major_formatter(LongitudeFormatter(number_format='.0f'))
        ax.yaxis.set_major_formatter(LatitudeFormatter(number_format='.0f'))
        ax.tick_params(labelsize=8, length=3, width=0.8, pad=2)
    else:
        ax.set_xticks([])
        ax.set_yticks([])

    draw_domain_border(ax)

    # 槽线
    lon_cl, lat_cl, _ = extract_ea_trough_line_and_index_optimized(
        hm_clim, EA_LON_MIN, EA_LON_MAX, EA_LAT_MIN, EA_LAT_MAX
    )
    if lon_cl is not None:
        ax.plot(lon_cl, lat_cl, transform=proj, color=CLIM_COLOR, lw=CLIM_LW, ls=CLIM_LS, zorder=40)

    lon_yr, lat_yr, _ = extract_ea_trough_line_and_index_optimized(
        hm_yr, EA_LON_MIN, EA_LON_MAX, EA_LAT_MIN, EA_LAT_MAX
    )
    if lon_yr is not None:
        ax.plot(lon_yr, lat_yr, transform=proj, color=YR_COLOR, lw=YR_LW, ls=YR_LS, zorder=41)

    ax.set_title(title_text, fontsize=10, pad=4)

# ================== 主图：6张平铺 ==================
def plot_flat(outfile):
    fig = plt.figure(figsize=(10.5, 6.0))
    axs = [fig.add_subplot(2, 3, i + 1, projection=ccrs.PlateCarree()) for i in range(6)]

    for i, (start, end) in enumerate(date_ranges):
        hm_yr   = mean_period_by_slice(hgt_2025, start, end)
        hm_clim = mean_period_by_monthday(hgt_clim_all, start, end)

        if hm_yr is None or hm_clim is None:
            axs[i].set_title("Empty", fontsize=9)
            continue

        plot_panel(
            axs[i], hm_yr, hm_clim,
            f"{start:%m-%d} – {end:%m-%d}",
            is_first_panel=(i == 0)
        )

    fig.suptitle(
        "MAM 500 hPa Geopotential Height (2025) & East Asian Trough Line\n"
        "1991–2020 Climatology (Dashed Red) vs 2025 (Solid Green)",
        fontsize=12, y=0.98
    )

    # 优化：移除 Colorbar，直接紧凑布局
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(outfile, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", outfile)

# ================== 运行 ==================
print("Loading HGT target year...")
hgt_2025 = open_hgt_year(year_target)

print("Loading HGT climatology years...")
hgt_clim_all = xr.concat([open_hgt_year(y) for y in years_clim], dim="time")

out = os.path.join(output_dir, "EAT_flat_HGT_trough_NoCbar.pdf")
plot_flat(out)

hgt_2025.close()
hgt_clim_all.close()
print("✅ 绘图完成！")
